# DEIT 2:4 sparsity

- fc1 layer은 fusion으로 묶여 처리되어 선택되지 않은것

Name: __myl_Fc_myl2_7
LayerType: fusion
Metadata: fc1/MatMul + GELU 관련 Mul, Add, Div, Erf + fc1/Add 등으로 합쳐서 tensorrt에서 처리하고 있었다.



### 1) Dynamic engine batch size별 비교
```python
# 이렇게 설정
profile.set_shape(
        input_name,
        min=(1, 3, 224, 224),
        opt=(64, 3, 224, 224),
        max=(256, 3, 224, 224)
    )
```

| Batch size | Dense mean ms | Sparse mean ms | Dense throughput | Sparse throughput | Speedup |
| ---------: | ------------: | -------------: | ---------------: | ----------------: | ------: |
|          8 |         15.71 |          14.56 |           509.23 |            549.39 |  1.079x |
|         16 |         29.61 |          27.45 |           540.37 |            582.84 |  1.079x |
|         32 |         56.76 |          52.72 |           563.81 |            606.93 |  1.076x |
|         64 |        107.38 |         100.61 |           596.01 |            636.13 |  1.067x |
|        128 |        220.03 |         204.63 |           581.75 |            625.53 |  1.075x |
|        256 |        438.57 |         409.26 |           583.71 |            625.52 |  1.072x |


In [ ]:
import tensorrt as trt
import numpy as np
import torch
import pandas as pd

# =========================
# 1. TensorRT Engine Build
# =========================

logger = trt.Logger(trt.Logger.VERBOSE)

def build_engine(onnx_path, sparse=False, max_batch_size=256):
    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)
        print("Sparse Weights 활성화!")

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())


    input_name = network.get_input(0).name
    print("Input name:", input_name)

    profile = builder.create_optimization_profile()

    profile.set_shape(
        input_name,
        min=(1, 3, 224, 224),
        opt=(64, 3, 224, 224),
        max=(max_batch_size, 3, 224, 224)
    )

    config.add_optimization_profile(profile)

    print(f"엔진 빌드 중... ({onnx_path})")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("TensorRT engine build failed")

    print("완료!")
    return engine_bytes


engine_dense = build_engine(
    "deit_onnx/deit_small_dense.onnx",
    sparse=False,
    max_batch_size=256
)

engine_sparse = build_engine(
    "deit_onnx/deit_small_sparse.onnx",
    sparse=True,
    max_batch_size=256
)

In [2]:
# =========================
# 2. TensorRT Inference Test
# =========================
import tensorrt as trt

runtime_logger = trt.Logger(trt.Logger.WARNING)

def run_engine(engine_bytes, batch_size=1, warmup=30, repeat=100):
    runtime = trt.Runtime(runtime_logger)
    engine = runtime.deserialize_cuda_engine(engine_bytes)
    context = engine.create_execution_context()


    input_name = None
    output_name = None

    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        mode = engine.get_tensor_mode(name)

        if mode == trt.TensorIOMode.INPUT:
            input_name = name
        elif mode == trt.TensorIOMode.OUTPUT:
            output_name = name

    print("Input:", input_name)
    print("Output:", output_name)

    input_shape = (batch_size, 3, 224, 224)

    input_data = np.random.randn(*input_shape).astype(np.float32)

    input_tensor = torch.from_numpy(input_data).to(
        device="cuda",
        dtype=torch.float32
    ).contiguous()

    context.set_input_shape(input_name, input_shape)

    output_shape = tuple(context.get_tensor_shape(output_name))

    output_tensor = torch.empty(
        output_shape,
        device="cuda",
        dtype=torch.float32
    )

    context.set_tensor_address(input_name, input_tensor.data_ptr())
    context.set_tensor_address(output_name, output_tensor.data_ptr())

    stream = torch.cuda.Stream()

    # 시간 측정 전 워밍업
    with torch.cuda.stream(stream):
        for _ in range(warmup):
            ok = context.execute_async_v3(stream.cuda_stream)
            if not ok:
                raise RuntimeError("TensorRT execution failed during warmup")

    torch.cuda.synchronize()

    times = []
    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    # 시간 측정
    with torch.cuda.stream(stream):
        for _ in range(repeat):
            starter.record(stream)

            ok = context.execute_async_v3(stream.cuda_stream)

            ender.record(stream)

            if not ok:
                raise RuntimeError("TensorRT execution failed during timing")

            ender.synchronize()
            times.append(starter.elapsed_time(ender))

    torch.cuda.synchronize()

    times = np.array(times)

    mean_ms = float(times.mean())
    median_ms = float(np.median(times))
    min_ms = float(times.min())
    p90_ms = float(np.percentile(times, 90))

    throughput = batch_size / (mean_ms / 1000)

    return {
        "batch_size": batch_size,
        "mean_ms": mean_ms,
        "median_ms": median_ms,
        "min_ms": min_ms,
        "p90_ms": p90_ms,
        "throughput_samples_per_sec": throughput,
        "output_shape": output_shape
    }

In [3]:
# =========================
# 3. Batch Size Test
# =========================

batch_sizes = [16, 64, 256]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        engine_dense,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        engine_sparse,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 16

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 29.609 ms, 중앙값: 29.606 ms, 최소: 29.563 ms, p90: 29.633 ms, throughput: 540.37 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 27.452 ms, 중앙값: 27.451 ms, 최소: 27.422 ms, p90: 27.469 ms, throughput: 582.84 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.079x
Median latency speedup    : 1.079x
Throughput speedup        : 1.079x

Batch size = 64

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 107.380 ms, 중앙값: 107.376 ms, 최소: 107.307 ms, p90: 107.412 ms, throughput: 596.01 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 100.608 ms, 중앙값: 100.605 ms, 최소: 100.539 ms, p90: 100.645 ms, throughput: 636.13 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.067x
Median latency speedup    : 1.067x
Throughput speedup        : 1.067x

Batch size = 256

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 438.575 ms, 중앙값: 438.569 ms, 최소: 438.445 ms, p90: 438.634 ms, thr

,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,16,29.609192,27.451572,29.606320,27.450624,540.372725,582.844578,1.078597,1.078530,1.078597
1,64,107.379923,100.607978,107.375744,100.604862,596.014585,636.132452,1.067310,1.067302,1.067310
2,256,438.574869,409.259367,438.569199,409.248428,583.708776,625.520198,1.071631,1.071645,1.071631


In [5]:
# =========================
# 3. Batch Size Test
# =========================

batch_sizes = [8, 32, 128]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        engine_dense,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        engine_sparse,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 8

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 15.710 ms, 중앙값: 15.708 ms, 최소: 15.689 ms, p90: 15.726 ms, throughput: 509.23 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 14.562 ms, 중앙값: 14.562 ms, 최소: 14.525 ms, p90: 14.575 ms, throughput: 549.39 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.079x
Median latency speedup    : 1.079x
Throughput speedup        : 1.079x

Batch size = 32

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 56.756 ms, 중앙값: 56.752 ms, 최소: 56.708 ms, p90: 56.790 ms, throughput: 563.81 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 52.725 ms, 중앙값: 52.721 ms, 최소: 52.684 ms, p90: 52.747 ms, throughput: 606.93 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.076x
Median latency speedup    : 1.076x
Throughput speedup        : 1.076x

Batch size = 128

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 220.027 ms, 중앙값: 220.026 ms, 최소: 219.947 ms, p90: 220.064 ms, throughput: 

,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,8,15.709942,14.561669,15.707920,14.561936,509.231668,549.387582,1.078856,1.078697,1.078856
1,32,56.756381,52.724526,56.751968,52.721008,563.813253,606.928166,1.076470,1.076458,1.076470
2,128,220.027442,204.625872,220.026382,204.618866,581.745616,625.531849,1.075267,1.075299,1.075267


### 2) Fixed size engine batch size별 비교
``` python
profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    ) # 이렇게 설정
```

| batch | dense mean (ms) | sparse mean (ms) | throughput speedup | mean speedup |
|-------|----------------|------------------|--------------------|--------------|
| 32    | 53.87          | 50.80            | 629.92 / 594.01    | 1.060x       |
| 64    | 105.32         | 98.49            | 649.84 / 607.65    | 1.069x       |
| 128   | 207.20         | 193.27           | 662.30 / 617.76    | 1.072x       |
| 256   | 428.90         | 404.54           | 632.82 / 596.87    | 1.060x       |

- BS=256

In [ ]:
def build_fixed_batch_engine(onnx_path, batch_size, sparse=True):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.VERBOSE)

    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)

    # 상세 layer/tactic 정보 확인용
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    input_name = network.get_input(0).name

    profile = builder.create_optimization_profile()
    fixed_shape = (batch_size, 3, 224, 224)

    profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    )
    config.add_optimization_profile(profile)

    print(f"Build fixed-shape engine: batch={batch_size}, sparse={sparse}")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    return engine_bytes

batch_size = 256

fixed_dense_engine_256 = build_fixed_batch_engine(
        "deit_onnx/deit_small_dense.onnx",
        batch_size=batch_size,
        sparse=False
    )

fixed_sparse_engine_256 = build_fixed_batch_engine(
        "deit_onnx/deit_small_sparse.onnx",
        batch_size=batch_size,
        sparse=True
    )

In [ ]:
import numpy as np
import torch
import pandas as pd

batch_sizes = [256]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        fixed_dense_engine_256,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        fixed_sparse_engine_256,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results

- BS=128

In [ ]:
def build_fixed_batch_engine(onnx_path, batch_size, sparse=True):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.VERBOSE)

    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)

    # 상세 layer/tactic 정보 확인용
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    input_name = network.get_input(0).name

    profile = builder.create_optimization_profile()
    fixed_shape = (batch_size, 3, 224, 224)

    profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    )
    config.add_optimization_profile(profile)

    print(f"Build fixed-shape engine: batch={batch_size}, sparse={sparse}")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    return engine_bytes

batch_size = 128

fixed_dense_engine_128 = build_fixed_batch_engine(
        "deit_onnx/deit_small_dense.onnx",
        batch_size=batch_size,
        sparse=False
    )

fixed_sparse_engine_128 = build_fixed_batch_engine(
        "deit_onnx/deit_small_sparse.onnx",
        batch_size=batch_size,
        sparse=True
    )

In [8]:
batch_sizes = [128]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        fixed_dense_engine_128,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        fixed_sparse_engine_128,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 128

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 207.199 ms, 중앙값: 207.196 ms, 최소: 207.057 ms, p90: 207.278 ms, throughput: 617.76 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 193.267 ms, 중앙값: 193.257 ms, 최소: 193.131 ms, p90: 193.343 ms, throughput: 662.30 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.072x
Median latency speedup    : 1.072x
Throughput speedup        : 1.072x


,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,128,207.198763,193.267143,207.196144,193.257423,617.764306,662.295711,1.072085,1.072125,1.072085


- BS=64

In [ ]:
def build_fixed_batch_engine(onnx_path, batch_size, sparse=True):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.VERBOSE)

    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)

    # 상세 layer/tactic 정보 확인용
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    input_name = network.get_input(0).name

    profile = builder.create_optimization_profile()
    fixed_shape = (batch_size, 3, 224, 224)

    profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    )
    config.add_optimization_profile(profile)

    print(f"Build fixed-shape engine: batch={batch_size}, sparse={sparse}")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    return engine_bytes

batch_size = 64

fixed_dense_engine_64 = build_fixed_batch_engine(
        "deit_onnx/deit_small_dense.onnx",
        batch_size=batch_size,
        sparse=False
    )     

fixed_sparse_engine_64 = build_fixed_batch_engine(
        "deit_onnx/deit_small_sparse.onnx",
        batch_size=batch_size,
        sparse=True
    )

In [10]:
batch_sizes = [64]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        fixed_dense_engine_64,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        fixed_sparse_engine_64,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 64

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 105.323 ms, 중앙값: 105.327 ms, 최소: 105.222 ms, p90: 105.384 ms, throughput: 607.65 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 98.486 ms, 중앙값: 98.486 ms, 최소: 98.356 ms, p90: 98.542 ms, throughput: 649.84 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.069x
Median latency speedup    : 1.069x
Throughput speedup        : 1.069x


,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,64,105.323337,98.485599,105.327251,98.485775,607.652605,649.841202,1.069429,1.069467,1.069429


- BS=32

In [ ]:
def build_fixed_batch_engine(onnx_path, batch_size, sparse=True):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.VERBOSE)

    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)

    # 상세 layer/tactic 정보 확인용
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    input_name = network.get_input(0).name

    profile = builder.create_optimization_profile()
    fixed_shape = (batch_size, 3, 224, 224)

    profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    )
    config.add_optimization_profile(profile)

    print(f"Build fixed-shape engine: batch={batch_size}, sparse={sparse}")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    return engine_bytes

batch_size = 32

fixed_dense_engine_32 = build_fixed_batch_engine(
        "deit_onnx/deit_small_dense.onnx",
        batch_size=batch_size,
        sparse=False                                        
    )

fixed_sparse_engine_32 = build_fixed_batch_engine(
        "deit_onnx/deit_small_sparse.onnx",
        batch_size=batch_size,
        sparse=True
    )

In [12]:
# =========================
# 3. Batch Size Test
# =========================

batch_sizes = [32]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        fixed_dense_engine_32,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        fixed_sparse_engine_32,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 32

Dense 엔진 측정 중...
Input: input
Output: output
Dense  평균: 53.871 ms, 중앙값: 53.871 ms, 최소: 53.817 ms, p90: 53.899 ms, throughput: 594.01 samples/s

Sparse 엔진 측정 중...
Input: input
Output: output
Sparse 평균: 50.800 ms, 중앙값: 50.797 ms, 최소: 50.731 ms, p90: 50.829 ms, throughput: 629.92 samples/s

=== 속도 향상 ===
Mean latency speedup      : 1.060x
Median latency speedup    : 1.061x
Throughput speedup        : 1.060x


,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,32,53.871298,50.799716,53.870672,50.797359,594.008338,629.924783,1.060465,1.060501,1.060465


### 3) fc1에 2:4 sparsity 적용해보기
- fusion을 풀려고 Identity도 추가해보고 , cast 도 해봤는데 tensorrt에서 최적화 하여 fusion이 풀리지는 않음
- 방법. fc1 output을 ONNX graph output으로 추가하기(dense, sparse속도가 더 느려지기는 할것) -> 적용됨

| batch | dense mean (ms) | sparse mean (ms) | dense median (ms) | sparse median (ms) | dense throughput | sparse throughput | mean speedup | median speedup |
|-------|----------------|------------------|-------------------|-------------------|-----------------|-----------------|--------------|----------------|
| 128   | 254.23         | 239.60           | 254.22            | 239.59            | 503.48          | 534.22          | 1.061x       | 1.061x         |

In [2]:
import onnx
model = onnx.load("deit_onnx/deit_small_sparse.onnx")
matmul_nodes = [n.name for n in model.graph.node if n.op_type == "MatMul"]
print(matmul_nodes)

['/blocks/blocks.0/attn/qkv/MatMul', '/blocks/blocks.0/attn/MatMul', '/blocks/blocks.0/attn/MatMul_1', '/blocks/blocks.0/attn/proj/MatMul', '/blocks/blocks.0/mlp/fc1/MatMul', '/blocks/blocks.0/mlp/fc2/MatMul', '/blocks/blocks.1/attn/qkv/MatMul', '/blocks/blocks.1/attn/MatMul', '/blocks/blocks.1/attn/MatMul_1', '/blocks/blocks.1/attn/proj/MatMul', '/blocks/blocks.1/mlp/fc1/MatMul', '/blocks/blocks.1/mlp/fc2/MatMul', '/blocks/blocks.2/attn/qkv/MatMul', '/blocks/blocks.2/attn/MatMul', '/blocks/blocks.2/attn/MatMul_1', '/blocks/blocks.2/attn/proj/MatMul', '/blocks/blocks.2/mlp/fc1/MatMul', '/blocks/blocks.2/mlp/fc2/MatMul', '/blocks/blocks.3/attn/qkv/MatMul', '/blocks/blocks.3/attn/MatMul', '/blocks/blocks.3/attn/MatMul_1', '/blocks/blocks.3/attn/proj/MatMul', '/blocks/blocks.3/mlp/fc1/MatMul', '/blocks/blocks.3/mlp/fc2/MatMul', '/blocks/blocks.4/attn/qkv/MatMul', '/blocks/blocks.4/attn/MatMul', '/blocks/blocks.4/attn/MatMul_1', '/blocks/blocks.4/attn/proj/MatMul', '/blocks/blocks.4/mlp/fc

In [13]:
import onnx
import onnx_graphsurgeon as gs
import numpy as np

input_path = "deit_onnx/deit_small_sparse.onnx"
output_path = "deit_onnx/deit_small_sparse_mark_fc1_outputs.onnx"

model = onnx.load(input_path)
graph = gs.import_onnx(model)

marked = 0

for node in graph.nodes:
    # TensorRT metadata에 보였던 fc1 Add 노드 기준
    # 예: /blocks/blocks.1/mlp/fc1/Add
    if "mlp/fc1/Add" in node.name or "mlp/fc1/Add" in str(node.outputs):
        for out in node.outputs:
            out.dtype = np.float16
            graph.outputs.append(out)
            marked += 1
            print("Marked as graph output:", out.name)

graph.cleanup().toposort()

onnx.save(gs.export_onnx(graph), output_path)

print("Saved:", output_path)
print("Marked fc1 outputs:", marked)


# ========================= dense 모델도 동일하게 fc1 출력 마킹 =========================
input_path = "deit_onnx/deit_small_dense.onnx"
output_path = "deit_onnx/deit_small_dense_mark_fc1_outputs.onnx"

model = onnx.load(input_path)
graph = gs.import_onnx(model)

marked = 0

for node in graph.nodes:
    # TensorRT metadata에 보였던 fc1 Add 노드 기준
    # 예: /blocks/blocks.1/mlp/fc1/Add
    if "mlp/fc1/Add" in node.name or "mlp/fc1/Add" in str(node.outputs):
        for out in node.outputs:
            out.dtype = np.float16
            graph.outputs.append(out)
            marked += 1
            print("Marked as graph output:", out.name)

graph.cleanup().toposort()

onnx.save(gs.export_onnx(graph), output_path)

print("Saved:", output_path)
print("Marked fc1 outputs:", marked)


Marked as graph output: /blocks/blocks.0/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.1/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.2/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.3/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.4/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.5/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.6/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.7/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.8/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.9/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.10/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.11/mlp/fc1/Add_output_0
Saved: deit_onnx/deit_small_sparse_mark_fc1_outputs.onnx
Marked fc1 outputs: 12
Marked as graph output: /blocks/blocks.0/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.1/mlp/fc1/Add_output_0
Marked as graph output: /blocks/blocks.2/mlp/fc1/A

In [14]:
def build_fixed_batch_engine(onnx_path, batch_size, sparse=True):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.VERBOSE)

    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()

    config.set_flag(trt.BuilderFlag.FP16)

    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)

    # 상세 layer/tactic 정보 확인용
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED

    with open(onnx_path, "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    input_name = network.get_input(0).name

    profile = builder.create_optimization_profile()
    fixed_shape = (batch_size, 3, 224, 224)

    profile.set_shape(
        input_name,
        min=fixed_shape,
        opt=fixed_shape,
        max=fixed_shape
    )
    config.add_optimization_profile(profile)

    print(f"Build fixed-shape engine: batch={batch_size}, sparse={sparse}")
    engine_bytes = builder.build_serialized_network(network, config)

    if engine_bytes is None:
        raise RuntimeError("Engine build failed")

    return engine_bytes

batch_size = 128

fixed_dense_engine_128 = build_fixed_batch_engine(
        "deit_onnx/deit_small_dense_mark_fc1_outputs.onnx",
        batch_size=batch_size,
        sparse=False                                        
    )

fixed_sparse_engine_128 = build_fixed_batch_engine(
        "deit_onnx/deit_small_sparse_mark_fc1_outputs.onnx",
        batch_size=batch_size,
        sparse=True
    )

[05/13/2026-14:57:15] [TRT] [I] The logger passed into createInferBuilder differs from one already provided for an existing builder, runtime, or refitter. Uses of the global logger, returned by nvinfer1::getLogger(), will return the existing value.
[05/13/2026-14:57:15] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 1257, GPU 16560 (MiB)
[05/13/2026-14:57:15] [TRT] [V] Trying to load shared library libnvinfer_builder_resource.so.10.3.0
[05/13/2026-14:57:15] [TRT] [V] Loaded shared library libnvinfer_builder_resource.so.10.3.0
[05/13/2026-14:57:17] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU +927, GPU +1, now: CPU 2184, GPU 16561 (MiB)
[05/13/2026-14:57:17] [TRT] [V] CUDA lazy loading is enabled.
[05/13/2026-14:57:17] [TRT] [V] Adding network input: input with dtype: float32, dimensions: (-1, 3, 224, 224)
[05/13/2026-14:57:17] [TRT] [V] Registering tensor: input for ONNX tensor: input
[05/13/2026-14:57:17] [TRT] [V] Importing initializer: patch_embed.proj

In [15]:
# =========================
# 2. TensorRT Inference Test
# =========================

runtime_logger = trt.Logger(trt.Logger.WARNING)

def trt_dtype_to_torch(dtype):
    if dtype == trt.float32:
        return torch.float32
    elif dtype == trt.float16:
        return torch.float16
    elif dtype == trt.int32:
        return torch.int32
    elif dtype == trt.int8:
        return torch.int8
    else:
        raise TypeError(f"Unsupported TensorRT dtype: {dtype}")


def run_engine(engine_bytes, batch_size=1, warmup=30, repeat=100):
    runtime = trt.Runtime(runtime_logger)
    engine = runtime.deserialize_cuda_engine(engine_bytes)
    context = engine.create_execution_context()

    input_name = None
    output_names = []

    # input / output 이름 찾기
    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        mode = engine.get_tensor_mode(name)

        if mode == trt.TensorIOMode.INPUT:
            input_name = name
        elif mode == trt.TensorIOMode.OUTPUT:
            output_names.append(name)

    print("Input:", input_name)
    print("Outputs:")
    for out_name in output_names:
        print(" -", out_name)

    # input shape 설정
    input_shape = (batch_size, 3, 224, 224)

    input_data = np.random.randn(*input_shape).astype(np.float32)

    input_tensor = torch.from_numpy(input_data).to(
        device="cuda",
        dtype=torch.float32
    ).contiguous()

    context.set_input_shape(input_name, input_shape)

    # input address 등록
    context.set_tensor_address(input_name, input_tensor.data_ptr())

    # 모든 output tensor 생성 및 address 등록
    output_tensors = {}
    output_shapes = {}

    for output_name in output_names:
        output_shape = tuple(context.get_tensor_shape(output_name))
        output_dtype = trt_dtype_to_torch(engine.get_tensor_dtype(output_name))

        output_tensor = torch.empty(
            output_shape,
            device="cuda",
            dtype=output_dtype
        )

        output_tensors[output_name] = output_tensor
        output_shapes[output_name] = output_shape

        context.set_tensor_address(output_name, output_tensor.data_ptr())

        print(
            f"Allocated output: {output_name}, "
            f"shape={output_shape}, dtype={output_dtype}"
        )

    stream = torch.cuda.Stream()

    # 시간 측정 전 워밍업
    with torch.cuda.stream(stream):
        for _ in range(warmup):
            ok = context.execute_async_v3(stream.cuda_stream)
            if not ok:
                raise RuntimeError("TensorRT execution failed during warmup")

    torch.cuda.synchronize()

    times = []
    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    # 시간 측정
    with torch.cuda.stream(stream):
        for _ in range(repeat):
            starter.record(stream)

            ok = context.execute_async_v3(stream.cuda_stream)

            ender.record(stream)

            if not ok:
                raise RuntimeError("TensorRT execution failed during timing")

            ender.synchronize()
            times.append(starter.elapsed_time(ender))

    torch.cuda.synchronize()

    times = np.array(times)

    mean_ms = float(times.mean())
    median_ms = float(np.median(times))
    min_ms = float(times.min())
    p90_ms = float(np.percentile(times, 90))

    throughput = batch_size / (mean_ms / 1000)

    return {
        "batch_size": batch_size,
        "mean_ms": mean_ms,
        "median_ms": median_ms,
        "min_ms": min_ms,
        "p90_ms": p90_ms,
        "throughput_samples_per_sec": throughput,
        "output_shapes": output_shapes,
        "output_names": output_names
    }

In [16]:
# =========================
# 3. Batch Size Test
# =========================

batch_sizes = [128]

results = []

for bs in batch_sizes:
    print("\n" + "=" * 50)
    print(f"Batch size = {bs}")
    print("=" * 50)

    print("\nDense 엔진 측정 중...")
    dense_result = run_engine(
        fixed_dense_engine_128,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Dense  평균: {dense_result['mean_ms']:.3f} ms, "
        f"중앙값: {dense_result['median_ms']:.3f} ms, "
        f"최소: {dense_result['min_ms']:.3f} ms, "
        f"p90: {dense_result['p90_ms']:.3f} ms, "
        f"throughput: {dense_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    print("\nSparse 엔진 측정 중...")
    sparse_result = run_engine(
        fixed_sparse_engine_128,
        batch_size=bs,
        warmup=30,
        repeat=100
    )

    print(
        f"Sparse 평균: {sparse_result['mean_ms']:.3f} ms, "
        f"중앙값: {sparse_result['median_ms']:.3f} ms, "
        f"최소: {sparse_result['min_ms']:.3f} ms, "
        f"p90: {sparse_result['p90_ms']:.3f} ms, "
        f"throughput: {sparse_result['throughput_samples_per_sec']:.2f} samples/s"
    )

    mean_speedup = dense_result["mean_ms"] / sparse_result["mean_ms"]
    median_speedup = dense_result["median_ms"] / sparse_result["median_ms"]
    throughput_speedup = (
        sparse_result["throughput_samples_per_sec"]
        / dense_result["throughput_samples_per_sec"]
    )

    print("\n=== 속도 향상 ===")
    print(f"Mean latency speedup      : {mean_speedup:.3f}x")
    print(f"Median latency speedup    : {median_speedup:.3f}x")
    print(f"Throughput speedup        : {throughput_speedup:.3f}x")

    results.append({
        "batch_size": bs,

        "dense_mean_ms": dense_result["mean_ms"],
        "sparse_mean_ms": sparse_result["mean_ms"],

        "dense_median_ms": dense_result["median_ms"],
        "sparse_median_ms": sparse_result["median_ms"],

        "dense_throughput": dense_result["throughput_samples_per_sec"],
        "sparse_throughput": sparse_result["throughput_samples_per_sec"],

        "mean_latency_speedup": mean_speedup,
        "median_latency_speedup": median_speedup,
        "throughput_speedup": throughput_speedup
    })

df_results = pd.DataFrame(results)
df_results


Batch size = 128

Dense 엔진 측정 중...
Input: input
Outputs:
 - output
 - /blocks/blocks.0/mlp/fc1/Add_output_0
 - /blocks/blocks.1/mlp/fc1/Add_output_0
 - /blocks/blocks.2/mlp/fc1/Add_output_0
 - /blocks/blocks.3/mlp/fc1/Add_output_0
 - /blocks/blocks.4/mlp/fc1/Add_output_0
 - /blocks/blocks.5/mlp/fc1/Add_output_0
 - /blocks/blocks.6/mlp/fc1/Add_output_0
 - /blocks/blocks.7/mlp/fc1/Add_output_0
 - /blocks/blocks.8/mlp/fc1/Add_output_0
 - /blocks/blocks.9/mlp/fc1/Add_output_0
 - /blocks/blocks.10/mlp/fc1/Add_output_0
 - /blocks/blocks.11/mlp/fc1/Add_output_0
Allocated output: output, shape=(128, 1000), dtype=torch.float32
Allocated output: /blocks/blocks.0/mlp/fc1/Add_output_0, shape=(128, 197, 1536), dtype=torch.float16
Allocated output: /blocks/blocks.1/mlp/fc1/Add_output_0, shape=(128, 197, 1536), dtype=torch.float16
Allocated output: /blocks/blocks.2/mlp/fc1/Add_output_0, shape=(128, 197, 1536), dtype=torch.float16
Allocated output: /blocks/blocks.3/mlp/fc1/Add_output_0, shape=(128, 1

,batch_size,dense_mean_ms,sparse_mean_ms,dense_median_ms,sparse_median_ms,dense_throughput,sparse_throughput,mean_latency_speedup,median_latency_speedup,throughput_speedup
0,128,254.231885,239.600178,254.224907,239.594803,503.477367,534.223308,1.061067,1.061062,1.061067
